![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 02: AI Modelling Basics)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- You are free to use, modify and distribute this package for teaching, learning and research purposes.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 2B: Deep Learning Image Classification

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A small PyTorch image classifier trained on synthetic image patterns</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m02b-overview)
2. [Setup and Background](#m02b-setup)
3. [Core Concepts](#m02b-core-concepts)
4. [Guided Implementation](#m02b-guided-implementation)
5. [Testing and Analysis](#m02b-testing)
6. [Student Tasks](#m02b-student-tasks)
7. [Submission and Reflection](#m02b-submission)

---

<a id="m02b-overview"></a>

### 1. Overview and Learning Goals

This session introduces deep learning image classification using a small synthetic image dataset. The aim is not to build a production computer-vision model. The aim is to understand the basic workflow: create image tensors, assign class labels, split data, define a neural network, train the model, evaluate accuracy and inspect errors.

In `M02A`, the model input was a single numeric feature. In this session, the model input is an image tensor. A tensor is a multi-dimensional array. For image classification, a tensor often has the shape `(channels, height, width)`. A neural network learns patterns from these tensors and maps them to class labels.

This notebook uses synthetic images rather than external image datasets. That choice keeps the practical small, reproducible and safe. Later, if the unit needs public data, it should use public unit materials first or public datasets from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

By the end of this lab, you should be able to explain what an image tensor is, describe the image classification pipeline, build a small PyTorch model, train it for a few epochs, compute accuracy, and test normal, edge and failure behaviours in an image-classification workflow.

<a id="m02b-setup"></a>

### 2. Setup and Background

This notebook uses PyTorch for neural network modelling and Matplotlib for visualisation. The dataset contains two synthetic classes.

<div align="center">

<table>
<thead>
<tr><th><strong>Class</strong></th><th><strong>Pattern</strong></th><th><strong>Label</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Vertical bar</td><td>A bright vertical stripe in the centre of a small image.</td><td><code>0</code></td></tr>
<tr><td align="left">Horizontal bar</td><td>A bright horizontal stripe in the centre of a small image.</td><td><code>1</code></td></tr>
</tbody>
</table>

</div>

The patterns are deliberately simple. This lets you inspect the image tensors and understand why a small neural network can learn the distinction.

In [ ]:
import random
from typing import Any, Dict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("Setup complete.")

<a id="m02b-core-concepts"></a>

### 3. Core Concepts

Image classification is a supervised learning task. The model receives an image and predicts a class label. The training data contains both the image tensor and the correct label.

A simple image-classification pipeline has the following steps.

<div align="center">

<table>
<thead>
<tr><th><strong>Step</strong></th><th><strong>Purpose</strong></th><th><strong>Output</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Create or load images</td><td>Prepare input tensors.</td><td><code>X</code> with shape <code>(N, C, H, W)</code>.</td></tr>
<tr><td align="left">Assign labels</td><td>Define the target class for each image.</td><td><code>y</code> with shape <code>(N,)</code>.</td></tr>
<tr><td align="left">Split data</td><td>Separate training and test examples.</td><td>Training and test tensors.</td></tr>
<tr><td align="left">Define model</td><td>Create a neural network.</td><td>Predicted class logits.</td></tr>
<tr><td align="left">Train model</td><td>Update model parameters using loss.</td><td>Lower training loss.</td></tr>
<tr><td align="left">Evaluate model</td><td>Measure performance on held-out data.</td><td>Accuracy and error analysis.</td></tr>
</tbody>
</table>

</div>

The model output before softmax is often called **logits**. For a two-class classifier, each image receives two logits. The class with the higher logit is the predicted class.

In [ ]:
def make_bar_image(image_size=16, label=0, noise_level=0.10):
    # Create a synthetic image with either a vertical or horizontal bright bar.
    # label=0 means vertical bar. label=1 means horizontal bar.
    if image_size <= 0:
        raise ValueError("image_size must be positive.")
    if label not in (0, 1):
        raise ValueError("label must be 0 or 1.")
    if noise_level < 0:
        raise ValueError("noise_level must not be negative.")

    image = np.random.normal(loc=0.0, scale=noise_level, size=(image_size, image_size)).astype(np.float32)
    centre = image_size // 2

    if label == 0:
        image[:, centre-1:centre+1] += 1.0
    else:
        image[centre-1:centre+1, :] += 1.0

    image = np.clip(image, 0.0, 1.0)
    return image


vertical = make_bar_image(label=0)
horizontal = make_bar_image(label=1)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(vertical, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Label 0: vertical")
axes[0].axis("off")
axes[1].imshow(horizontal, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Label 1: horizontal")
axes[1].axis("off")
plt.show()

The two images show the two classes. The vertical-bar class has a bright central vertical stripe, while the horizontal-bar class has a bright central horizontal stripe. The added noise makes the task slightly less trivial while remaining easy enough for a small model.

<a id="m02b-guided-implementation"></a>

### 4. Guided Implementation

We now create a dataset, split it into training and test sets, define a small neural network and train it. The network is intentionally small so that the training runs quickly in Colab or local Jupyter.

The image tensor shape will be `(N, C, H, W)`, where `N` is the number of images, `C=1` is the grayscale channel count, and `H=W=16` are image height and width.

In [ ]:
def create_synthetic_bar_dataset(n_per_class=80, image_size=16, noise_level=0.10):
    # Create a balanced two-class image dataset.
    if n_per_class <= 0:
        return {"ok": False, "error": "n_per_class must be positive.", "result": None}

    images = []
    labels = []

    for label in [0, 1]:
        for _ in range(n_per_class):
            img = make_bar_image(image_size=image_size, label=label, noise_level=noise_level)
            images.append(img)
            labels.append(label)

    X = np.stack(images).astype(np.float32)
    y = np.array(labels, dtype=np.int64)

    X = X[:, None, :, :]  # Add channel dimension: (N, H, W) -> (N, C, H, W)

    indices = np.arange(len(y))
    np.random.shuffle(indices)

    X = X[indices]
    y = y[indices]

    return {"ok": True, "error": None, "result": (torch.tensor(X), torch.tensor(y))}


dataset_result = create_synthetic_bar_dataset(n_per_class=80)
dataset_result["ok"], dataset_result["result"][0].shape, dataset_result["result"][1].shape

The output should show `ok=True`. The image tensor should have shape `(160, 1, 16, 16)`, meaning 160 images, one grayscale channel, and 16 by 16 pixels. The label tensor should have shape `(160,)`.

In [ ]:
def split_tensors(X, y, test_ratio=0.25):
    # Split tensors into training and test parts.
    if not isinstance(X, torch.Tensor) or not isinstance(y, torch.Tensor):
        return {"ok": False, "error": "X and y must be torch tensors.", "result": None}

    if len(X) != len(y):
        return {"ok": False, "error": "X and y must have the same number of examples.", "result": None}

    if len(X) < 4:
        return {"ok": False, "error": "At least four examples are required.", "result": None}

    if not (0 < test_ratio < 1):
        return {"ok": False, "error": "test_ratio must be between 0 and 1.", "result": None}

    n_test = int(len(X) * test_ratio)
    if n_test == 0 or n_test >= len(X):
        return {"ok": False, "error": "Invalid test split size.", "result": None}

    X_train, X_test = X[:-n_test], X[-n_test:]
    y_train, y_test = y[:-n_test], y[-n_test:]

    return {
        "ok": True,
        "error": None,
        "result": (X_train, X_test, y_train, y_test)
    }


X, y = dataset_result["result"]
split_result = split_tensors(X, y, test_ratio=0.25)
X_train, X_test, y_train, y_test = split_result["result"]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

The split creates training and test tensors. The model will learn from the training tensors and will be evaluated on the test tensors. This is the image-classification analogue of the regression train/test split from M02A.

In [ ]:
class TinyCNN(nn.Module):
    # A very small CNN for two-class synthetic image classification.
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 2)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = TinyCNN()
sample_logits = model(X_train[:4])
print("Sample logits shape:", sample_logits.shape)
print(sample_logits)

The sample logits have shape `(4, 2)`, meaning four input images and two class scores for each image. The logits are not probabilities. The training loss will compare these logits with the correct labels.

In [ ]:
def train_classifier(model, X_train, y_train, epochs=8, batch_size=16, learning_rate=0.01):
    # Train a small classifier and return training history.
    if epochs <= 0:
        return {"ok": False, "error": "epochs must be positive.", "result": None}
    if batch_size <= 0:
        return {"ok": False, "error": "batch_size must be positive.", "result": None}

    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    history = []
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0
        total_examples = 0

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(xb)
            total_examples += len(xb)

        avg_loss = total_loss / total_examples
        history.append(avg_loss)
        print(f"Epoch {epoch + 1:02d} | loss = {avg_loss:.4f}")

    return {"ok": True, "error": None, "result": history}


train_result = train_classifier(model, X_train, y_train, epochs=8)

The training output shows the average loss for each epoch. The loss should generally decrease. It may not decrease perfectly every epoch, especially with small data and random initialisation, but the overall trend should improve if the model is learning.

In [ ]:
def compute_accuracy(model, X_eval, y_eval):
    # Compute classification accuracy.
    if not isinstance(X_eval, torch.Tensor) or not isinstance(y_eval, torch.Tensor):
        return {"ok": False, "error": "X_eval and y_eval must be torch tensors.", "result": None}

    if len(X_eval) != len(y_eval):
        return {"ok": False, "error": "X_eval and y_eval must have the same number of examples.", "result": None}

    if len(X_eval) == 0:
        return {"ok": False, "error": "Evaluation data must not be empty.", "result": None}

    model.eval()
    with torch.no_grad():
        logits = model(X_eval)
        predictions = torch.argmax(logits, dim=1)
        accuracy = (predictions == y_eval).float().mean().item()

    return {
        "ok": True,
        "error": None,
        "result": {
            "accuracy": accuracy,
            "predictions": predictions
        }
    }


train_accuracy = compute_accuracy(model, X_train, y_train)
test_accuracy = compute_accuracy(model, X_test, y_test)

print("Train accuracy:", train_accuracy["result"]["accuracy"])
print("Test accuracy:", test_accuracy["result"]["accuracy"])

The test accuracy estimates how well the model performs on held-out images. Because the synthetic patterns are simple, the accuracy should usually be high. High accuracy here does not mean the model is generally powerful; it means the model learned this simple two-pattern task.

In [ ]:
preds = test_accuracy["result"]["predictions"]

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"T:{y_test[i].item()} P:{preds[i].item()}")
    ax.axis("off")
plt.show()

Each title shows the true label (`T`) and predicted label (`P`). This inspection is useful because a single accuracy number does not show which examples were correct or incorrect. Later evaluation practicals will expand this idea to richer error analysis.

<a id="m02b-testing"></a>

### 5. Testing and Analysis

We now test key components of the workflow. The tests check shape expectations, safe rejection of invalid inputs and evaluation behaviour.

<div align="center">

<table>
<thead>
<tr><th><strong>Test type</strong></th><th><strong>Purpose</strong></th><th><strong>Example</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Normal</td><td>Check expected dataset and model behaviour.</td><td>Dataset has image tensor and label tensor.</td></tr>
<tr><td align="left">Edge</td><td>Check minimal valid conditions.</td><td>Very small balanced dataset still creates tensors.</td></tr>
<tr><td align="left">Failure</td><td>Reject invalid input safely.</td><td>Invalid label, invalid split ratio, mismatched tensor lengths.</td></tr>
</tbody>
</table>

</div>

In [ ]:
normal_data = create_synthetic_bar_dataset(n_per_class=4)
assert normal_data["ok"] is True
normal_X, normal_y = normal_data["result"]
assert normal_X.ndim == 4
assert normal_X.shape[1:] == (1, 16, 16)
assert normal_y.ndim == 1

edge_data = create_synthetic_bar_dataset(n_per_class=1)
assert edge_data["ok"] is True
edge_X, edge_y = edge_data["result"]
assert len(edge_X) == 2

bad_data = create_synthetic_bar_dataset(n_per_class=0)
assert bad_data["ok"] is False

bad_split = split_tensors(X, y, test_ratio=1.5)
assert bad_split["ok"] is False

bad_accuracy = compute_accuracy(model, X_test[:3], y_test[:2])
assert bad_accuracy["ok"] is False

print("Image classification pipeline tests passed.")

If this cell prints `Image classification pipeline tests passed.`, the main workflow components satisfy the specified behaviours. As before, passing tests provide evidence for these controlled cases, not a guarantee of correctness under all possible inputs.

<a id="m02b-student-tasks"></a>

### 6. Student Tasks

Extend the classifier by adding a third synthetic class: a diagonal bar. You should update the dataset generator, model output size and tests accordingly.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Create images for label <code>2</code> with a diagonal pattern.</td><td>Practises extending a dataset.</td><td>At least one visual example of the diagonal class.</td></tr>
<tr><td align="left">Task 2</td><td>Create a three-class dataset.</td><td>Practises multi-class classification.</td><td>Image tensor and labels include classes 0, 1 and 2.</td></tr>
<tr><td align="left">Task 3</td><td>Modify the model output size to 3.</td><td>Matches the number of classes.</td><td>Logits shape is <code>(batch_size, 3)</code>.</td></tr>
<tr><td align="left">Task 4</td><td>Train and evaluate the model.</td><td>Checks whether the classifier learns the new class.</td><td>Training loss and test accuracy.</td></tr>
<tr><td align="left">Task 5</td><td>Run normal, edge and failure checks.</td><td>Preserves robust workflow design.</td><td>Evidence that checks pass.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Implement a diagonal pattern and extend the dataset to three classes.
#
# Suggested direction:
# - label 0: vertical bar
# - label 1: horizontal bar
# - label 2: diagonal bar
#
# You may write new functions or modify the existing dataset generator.

# TODO: implement make_three_class_image(...)
# TODO: implement create_three_class_dataset(...)
# TODO: define a model with output size 3
# TODO: train and evaluate the three-class classifier

<a id="m02b-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Required item</strong></th><th><strong>What to submit</strong></th><th><strong>Quality check</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Three-class dataset</td><td>Images and labels for vertical, horizontal and diagonal classes.</td><td>Tensor shapes and label values are shown.</td></tr>
<tr><td align="left">Updated model</td><td>A classifier with output size 3.</td><td>Sample logits have shape <code>(batch_size, 3)</code>.</td></tr>
<tr><td align="left">Training and evaluation</td><td>Loss history and test accuracy.</td><td>Model trains without unexpected errors.</td></tr>
<tr><td align="left">Testing</td><td>Normal, edge and failure checks.</td><td>Invalid inputs are rejected or handled safely.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Reflection explains what changed when moving from two classes to three.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. What changed when the task moved from two classes to three classes?
2. Why must the model output size match the number of classes?
3. What does accuracy tell you in this synthetic example?
4. Why should we inspect individual predictions as well as summary metrics?
5. How does this image-classification pipeline relate to later multimodal and model-adaptation practicals?

#### Further Readings

- PyTorch tutorials: <https://pytorch.org/tutorials/>
- PyTorch neural networks: <https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html>
- PyTorch image classification tutorial: <https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>